<a href="https://colab.research.google.com/github/nishant-tyagi-ainishant-tyagi-ai/chromadb-rag-pipeline/blob/main/hyde_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain langchain-community chromadb sentence-transformers groq langchain-groq

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [14]:
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
print("Key set!")

Key set!


In [15]:
from langchain_groq import ChatGroq
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# LLM setup
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

# Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Setup done!")

/tmp/ipykernel_11235/1806265910.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Setup done!


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

docs = [
    Document(page_content="RAG stands for Retrieval Augmented Generation. It combines a retriever and a generator to answer questions using external knowledge."),
    Document(page_content="ChromaDB is a vector database used to store and retrieve embeddings efficiently for semantic search."),
    Document(page_content="HyDE stands for Hypothetical Document Embeddings. It generates a fake answer to a query and uses its embedding for retrieval."),
    Document(page_content="LangChain is a framework for building LLM-powered applications including RAG pipelines, agents, and chatbots."),
    Document(page_content="Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings."),
    Document(page_content="Query expansion improves retrieval by generating multiple variations of the original query to capture different perspectives."),
]

vectorstore = Chroma.from_documents(docs, embeddings, collection_name="hyde_demo")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Vectorstore ready!")

Vectorstore ready!


In [20]:
hyde_prompt = ChatPromptTemplate.from_template("""
You are an expert. Given the question below, write a short hypothetical passage
that would directly answer this question. Write only the passage, nothing else.

Question: {question}
Hypothetical Answer:""")

hyde_chain = hyde_prompt | llm | StrOutputParser()

answer_prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below.

Context: {context}
Question: {question}
Answer:""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def hyde_retrieve(question):
    hypothetical_doc = hyde_chain.invoke({"question": question})
    print(f"Hypothetical Doc:\n{hypothetical_doc}\n")
    retrieved_docs = retriever.invoke(hypothetical_doc)
    return retrieved_docs

rag_chain = (
    {"context": lambda x: format_docs(hyde_retrieve(x)), "question": RunnablePassthrough()}
    | answer_prompt
    | llm
    | StrOutputParser()
)

print("HyDE chain ready!")

HyDE chain ready!


In [21]:
result = rag_chain.invoke("What is HyDE and how does it improve retrieval?")
print(result)

Hypothetical Doc:
HyDE, or Hybrid Dense Embeddings, is a novel technique designed to enhance the efficiency and accuracy of information retrieval systems. By combining the strengths of sparse and dense embeddings, HyDE creates a more comprehensive and nuanced representation of data, allowing for more precise and relevant search results. This hybrid approach improves retrieval by capturing both the semantic relationships between entities and the subtle contextual differences that often get lost in traditional embedding methods. As a result, HyDE enables search systems to better understand the intent behind user queries, retrieve more accurate information, and ultimately provide a more satisfying user experience.

HyDE stands for Hypothetical Document Embeddings. It improves retrieval by generating a fake answer to a query and using its embedding for retrieval, allowing for more efficient semantic search, potentially in conjunction with a vector database like ChromaDB.


In [22]:
result2 = rag_chain.invoke("What is ChromaDB used for?")
print(result2)

Hypothetical Doc:
ChromaDB is a cutting-edge, open-source database management system specifically designed for high-performance and scalable storage of large-scale chromatography and mass spectrometry data. It is used by researchers and scientists in various fields, including proteomics, metabolomics, and genomics, to efficiently manage, analyze, and share complex experimental data. By providing a robust and flexible data management framework, ChromaDB enables users to streamline their data workflows, facilitate collaboration, and gain deeper insights into their research findings.

ChromaDB is used to store and retrieve embeddings efficiently for semantic search.
